# ReliefForge — dataset exploration
Runs locally on the small generated set (`data/smoke_ds`) or on the full dataset.
Shows the scene families, render styles and height conventions used by the report.

In [ ]:
import sys, json, random
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np, cv2
import matplotlib.pyplot as plt
%matplotlib inline
DS = ROOT / 'data' / 'smoke_ds'
assert (DS / 'scenes.jsonl').exists(), 'run src/data_download.py --out data/smoke_ds --n-scenes 24 first'
rows = [json.loads(l) for l in open(DS / 'scenes.jsonl')]
print(len(rows), 'scenes')
rows[0]

In [ ]:
from collections import Counter
print('families:', Counter(r['family'] for r in rows))
print('aspects:', Counter(r['aspect'] for r in rows))
print('splits:', Counter(r['split'] for r in rows))
print('depth mm: min %.1f max %.1f' % (min(r['relief_depth_mm_scene'] for r in rows), max(r['relief_depth_mm_scene'] for r in rows)))

In [ ]:
def show(scene_row, styles=('wood', 'clay', 'flat')):
    gid = f"{scene_row['id']:06d}"
    h = cv2.imread(str(DS / 'geometry' / scene_row['split'] / f'{gid}_h.png'), cv2.IMREAD_UNCHANGED).astype(np.float32) / 65535
    cols = [h]
    titles = ['height h in [0,1] (0 = base)']
    for st in styles:
        img = cv2.imread(str(DS / 'renders' / scene_row['split'] / f'{gid}_{st}.png'))
        cols.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img is not None else np.zeros_like(h))
        titles.append(st)
    fig, ax = plt.subplots(1, len(cols), figsize=(4 * len(cols), 4))
    for a, c, t in zip(ax, cols, titles):
        a.imshow(c, cmap='turbo' if c.ndim == 2 else None)
        a.set_title(t)
        a.axis('off')
    plt.suptitle(f"{scene_row['family']} {scene_row['W']}x{scene_row['H']} fg={scene_row['fg_frac']:.2f}")
    plt.show()

for fam in ('floral', 'botanical', 'scroll', 'border', 'medallion'):
    r = next((x for x in rows if x['family'] == fam and x['split'] != 'train'), None) or next(x for x in rows if x['family'] == fam)
    show(r)

In [ ]:
# height statistics over the split
stds, fgs = [], []
for r in rows:
    if r['split'] == 'train':
        stds.append(r['h_std']); fgs.append(r['fg_frac'])
plt.scatter(fgs, stds, s=12)
plt.xlabel('foreground fraction'); plt.ylabel('height std'); plt.title('train scenes')
plt.show()